In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Must install separately since Colab has torch 2.2.1/2.3.0+
if major_version >= 8:
    # Use this for new GPUs like Ampere, Hopper (A100, H100, L4)
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # Use this for older GPUs like Tesla T4, V100
    !pip install --no-deps "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# 1. Install the Zoo first
!pip install --upgrade unsloth_zoo

# 2. Install Unsloth based on your GPU
if major_version >= 8:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install --no-deps "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"

# 3. Ensure all other critical components are present
!pip install --upgrade xformers trl peft accelerate bitsandbytes

In [ ]:
%%capture
# 1. Uninstall existing versions to avoid metadata conflicts
!pip uninstall unsloth unsloth_zoo -y

# 2. Reinstall both directly from GitHub (source) to ensure synchronization
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth-zoo.git

# 3. Ensure core dependencies are updated to support the 2026 patches
!pip install --upgrade bitsandbytes accelerate xformers trl peft

In [ ]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import json

from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

In [ ]:
import pandas as pd

In [ ]:
training_set = pd.read_excel("/content/drive/MyDrive/Validation_AI_POC/fine_Tune_Data.xlsx")

In [ ]:
validation_set = pd.read_excel("/content/drive/MyDrive/Validation_AI_POC/val.xlsx")

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:

def prompt_formatter(example, prompt_template):
    instruction="""You are a QA Engineer. Your task is to develop test cases based on the given requirements.
NO PROSE: Do not provide explanations, assumptions, or "Additional Information" sections.
STRICT CONSTRAINTS: 1.For every signal listed under "Signals under test", you must generate at least one dedicated test case.
Example: If there are 3 signals under test, you must generate at least 3 test cases (TC_1, TC_2, TC_3) to ensure full coverage.
2. To generate the test cases you can only use:
  - signals under test (Example: i_can_DriveModeSelector)
  - Involved input signals (Example: i_can_DriveModeSelector)
  - The following keywords: "IF", "AND", "OR", "THEN"
  - The following operators: "==", "!=", "<", ">", "<=", ">="""

    requirement=example["reqs"]
    tc=example["tc"]

    formatted_prompt = tokenizer.bos_token + prompt_template.format(instruction, requirement, tc) + tokenizer.eos_token

    return {'text': formatted_prompt}

In [ ]:
from datasets import Dataset

In [ ]:
hf_train = Dataset.from_pandas(training_set)
hf_val = Dataset.from_pandas(validation_set)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [ ]:
tokenizer.add_bos_token, tokenizer.add_eos_token

(False, False)

In [ ]:
formatted_validation_dataset = hf_val.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

In [ ]:
formatted_training_dataset = hf_train.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
    random_state=42,
    loftq_config=None
)

Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from trl import SFTTrainer, SFTConfig # Add SFTConfig here
from transformers import TrainingArguments, EarlyStoppingCallback

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_training_dataset,
    eval_dataset = formatted_validation_dataset,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
    args = SFTConfig(
        max_seq_length = 4096,
        dataset_num_proc = 2,
        dataset_text_field = "text",
        packing = True,  # <--- Change this to True

        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 20,
        eval_strategy = "epoch",
        save_strategy = 'epoch',
        metric_for_best_model = "eval_loss",
        load_best_model_at_end = True,
        greater_is_better = False,
        learning_rate = 5e-5,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/64 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/64 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/9 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [ ]:
training_history = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14 | Num Epochs = 20 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 7,262,703,616 (0.29% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.312809,1.247661
2,1.119259,1.126865
3,1.085099,0.974219
4,0.821033,0.858803
5,0.883713,0.774111
6,0.599895,0.698951
7,0.581411,0.633780
8,0.427795,0.582951
9,0.462387,0.551723
10,0.294227,0.539922


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-2.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-4/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-4.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-6/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-6.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-8/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-8.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-10/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-10.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-12/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-12.
Unsloth: Restored added_token

In [ ]:
# @title Setup to enable bash commands
import locale

def getpreferredencoding():
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [ ]:
lora_model_name = "/content/drive/MyDrive/Validation_AI_POC/Mistral_TC_gen_2"

In [ ]:
model.save_pretrained(lora_model_name)

In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only.


('/content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only/tokenizer_config.json',
 '/content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only/chat_template.jinja',
 '/content/drive/MyDrive/Validation_AI_POC/Mistral_Adapters_Only/tokenizer.json')

In [ ]:
!ls -lh {lora_model_name}

total 81M
-rw------- 1 root root 1.3K May 11 08:33 adapter_config.json
-rw------- 1 root root  81M May 11 08:33 adapter_model.safetensors
-rw------- 1 root root 5.2K May 11 08:33 README.md


In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
instruction = """You are a QA Engineer. Your task is to develop test cases based on the given requirements.
NO PROSE: Do not provide explanations, assumptions, or "Additional Information" sections.
STRICT CONSTRAINTS: 1.For every signal listed under "Signals under test", you must generate at least one dedicated test case.
Example: If there are 3 signals under test, you must generate at least 3 test cases (TC_1, TC_2, TC_3) to ensure full coverage.
2. To generate the test cases you can only use:
  - signals under test (Example: i_can_DriveModeSelector)
  - Involved input signals (Example: i_can_DriveModeSelector)
  - Involved input parameters (Example: p_tng_lka_min_speed_activation)
  - The following keywords: "IF", "AND", "OR", "THEN"
  - The following operators: "==", "!=", "<", ">", "<=", ">="""

In [ ]:
test_REQ = """"6.3.2-2 For ACC Feature :

WHEN
o_can_adas_longi_state_comfort is set to  ""2"":""ALSCT_NOREQUEST_STANDBY""
AND
i_can_VehicleSpeed is between p_tng_kph_accoff_to_acc_speed_th_low  and p_tng_kph_acc_to_accoff_speed_thd_high

IF
(i_can_SetPlusSwitche = SET/+
OR
i_can_SetMinusSwitche = SET/-)
AND
i_can_VehicleSpeed > = p_tng_kph_acc_vmin_regulation


THEN
o_can_adas_longi_state_comfort  shall switch to ""3"":""ALSCT_INREGULATION""
AND
o_sdvipc_setspeed shall be set to i_can_VehicleSpeed

Note :   default value for p_tng_kph_acc_vmin_regulation  to consider for  automatic gearbox is 0 "
"""

In [ ]:
inputs = tokenizer(
[
    tokenizer.bos_token + alpaca_prompt.format(
        instruction,
        test_REQ,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")

In [ ]:
tokenizer.bos_token

'<s>'

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    use_cache=True,
    do_sample = False,
    pad_token_id=tokenizer.eos_token_id
)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [ ]:
print(
    tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
)

TC_1:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed >= 15
AND
i_can_SetPlusSwitche == SET/+
THEN
o_can_adas_longi_state_comfort == ALSC_INREGULATION(3)

TC_2:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed >= 15
AND
i_can_SetMinusSwitche == SET/-
THEN
o_can_adas_longi_state_comfort == ALSC_INREGULATION(3)

TC_3:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed >= 15
AND
i_can_SetPlusSwitche == SET/+
THEN
o_sdvipc_setspeed == i_can_VehicleSpeed

TC_4:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed >= 15
AND
i_can_SetMinusSwitche == SET/-
THEN
o_sdvipc_setspeed == i_can_VehicleSpeed


In [ ]:
validation_set

,reqs,tc
0,\nSignals under test:\no_sdvipc_AdasCarouselCo...,IF\ni_can_CarouselModeSwitch == 1(Pressed)\nTH...
1,\nSignals under test:\no_sdvipc_ADASCarouselMo...,TC_1:\nIF\ni_can_CarouselModeSwitch == 1(Press...
2,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 2...
3,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 4...
4,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\nACC_Status == SUSPENDED\nAND\no_can...
5,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 2...
6,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 2...
7,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 2...
8,\nSignals under test:\no_can_adas_longi_state_...,TC_1:\nIF\no_can_adas_longi_state_comfort == 2...


In [ ]:
reqs = validation_set['reqs'].to_list()

In [ ]:
tcs =[]
for i in reqs:
  inputs = tokenizer(
[
    tokenizer.bos_token + alpaca_prompt.format(
        instruction,
        i,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")
  outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    use_cache=True,
    do_sample = False,
    pad_token_id=tokenizer.eos_token_id
)
  tc =tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
  tcs.append(tc)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

In [ ]:
for i in tcs:
  print(i)
  print("_"*30)

TC_1:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_ALL_FEATURE_OFF(1)

TC_2:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_SPEED_LIMITER(2)

TC_3:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_ADAPTIVE_CRUISE_CONTROL(4)
______________________________
TC_1:
IF
i_can_CarouselModeSwitch == 1(Pressed)
AND
ACC is available
THEN
o_sdvipc_ADASCarouselModeSelected == 1(CF_ALL_FEATURE_OFF)

TC_2:
IF
i_can_CarouselModeSwitch == 1(Pressed)
AND
ACC is available
THEN
o_sdvipc_ADASCarouselModeSelected == 4(CF_ADAPTIVE_CRUISE_CONTROL)

TC_3:
IF
i_can_CarouselModeSwitch == 1(Pressed)
AND
ACC is available
THEN
o_sdvipc_ADASCarouselModeSelected == 2(CF_SPEED_LIMITER)
______________________________
TC_1:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_i_can_TCS_Deactivated == Deactivated
THEN
o_can_adas_longi_state_comfort == OFF

TC_2:
IF
o_can_adas_longi_state_comfort ==

In [ ]:
# generated with do not assume condition added
test_req =""""
Signals under test:
o_can_adas_longi_state_comfort, o_sdvipc_setspeed, AutoRestart_Status

Involved input signals:
o_can_adas_longi_state_comfort , i_can_VehicleSpeed, i_can_gearLeverRequested, TSR_OSP_SpeedLimit, i_can_AdaptSpeedSwitch, i_can_PressedByDriver

Involved input parameters:
p_tng_kph_acc_vmin_activation_on_brake, p_tng_acc_min_regulation_speed

Requirement:
6.3.2-9 WHEN
o_can_adas_longi_state_comfort has value = ""2"":""ALSCT_NOREQUEST_STANDBY""  (ACC_status : SUSPENDED OR WAITING)
AND
i_can_VehicleSpeed < p_tng_kph_acc_vmin_activation_on_brake (default = 0.7 kph, tunable)
AND
i_can_gearLeverRequested is in Drive or Brake mode
AND
TSR_OSP_SpeedLimit ≥ p_tng_acc_min_regulation_speed (30 kph default, tunable)

IF i_can_AdaptSpeedSwitch is ""pressed""
AND
i_can_PressedByDriver = Brake Pedal Not Pressed

THEN
o_can_adas_longi_state_comfort shall switch to INREGULATION
AND
o_sdvipc_setspeed shall be set to :TSR_OSP_SpeedLimit value

ELSEIF i_can_ContextSwitch is ""pressed""
AND
i_can_PressedByDriver = Brake Pedal Pressed

THEN
o_can_adas_longi_state_comfort shall switch to INSTOP
AND
A-CC-SL system shall set AutoRestart_Status to Manual Restart
AND
o_sdvipc_setspeed shall be set to :TSR_OSP_SpeedLimit value

Note:  AutorestartStatus (obs::accAutorestartStatus) is not accessible at PCU boundaries. However, the conditions and the CarMaker scenario used to set this signal from PCU boundaries are detailed in ""ACC_SyRS_1274"".
"""

In [ ]:
inputs = tokenizer(
[
    tokenizer.bos_token + alpaca_prompt.format(
        instruction,
        test_req,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    use_cache=True,
    do_sample = False,
    pad_token_id=tokenizer.eos_token_id
)
tc =tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [ ]:
print(tc)

TC_1:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed < 7
AND
i_can_gearLeverRequested == DRIVE
AND
TSR_OSP_SpeedLimit >= 30
AND
i_can_AdaptSpeedSwitch == PRESSED
AND
i_can_PressedByDriver == NOT_PRESSED
THEN
o_can_adas_longi_state_comfort == INREGULATION

TC_2:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed < 7
AND
i_can_gearLeverRequested == DRIVE
AND
TSR_OSP_SpeedLimit >= 30
AND
i_can_AdaptSpeedSwitch == PRESSED
AND
i_can_PressedByDriver == NOT_PRESSED
THEN
o_sdvipc_setspeed == TSR_OSP_SpeedLimit

TC_3:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed < 7
AND
i_can_gearLeverRequested == DRIVE
AND
TSR_OSP_SpeedLimit >= 30
AND
i_can_AdaptSpeedSwitch == PRESSED
AND
i_can_PressedByDriver == NOT_PRESSED
THEN
AutoRestart_Status == MANUAL_RESTART

TC_4:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_VehicleSpeed < 7
AND
i_can_gearLeverRequested == DRIV

In [ ]:
instruction = """You are a QA Engineer. Your task is to develop test cases based on the given requirements.
NO PROSE: Do not provide explanations, assumptions, or "Additional Information" sections.
STRICT CONSTRAINTS: 1.For every signal listed under "Signals under test", you must generate at least one dedicated test case.
Example: If there are 3 signals under test, you must generate at least 3 test cases (TC_1, TC_2, TC_3) to ensure full coverage.
2. To generate the test cases you can only use:
  - signals under test (Example: i_can_DriveModeSelector)
  - Involved input signals (Example: i_can_DriveModeSelector)
  - Involved input parameters (Example: p_tng_lka_min_speed_activation)
  - The following keywords: "IF", "AND", "OR", "THEN"
  - The following operators: "==", "!=", "<", ">", "<=", ">=
3. DO NOT assume any values. Only use whatever is given in the Input"""


In [ ]:
tcs =[]
for i in reqs:
  inputs = tokenizer(
[
    tokenizer.bos_token + alpaca_prompt.format(
        instruction,
        i,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")
  outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
)
  tc =tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
  tcs.append(tc)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

In [ ]:
a = 1
for i in tcs:
  print(a)
  print(i)
  print("_"*30)
  a = a+1

1
TC_1:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_ALL_FEATURE_OFF(1)

TC_2:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_SPEED_LIMITER(2)

TC_3:
IF
i_can_CarouselModeSwitch == 1(Pressed)
THEN
o_sdvipc_AdasCarouselContent == CF_ADAPTIVE_CRUISE_CONTROL(4)
______________________________
2
TC_1:
IF
i_can_CarouselModeSwitch == 1(Pressed)
AND
o_sdvipc_ADASCarouselModeSelected == 1(CF_ALL_FEATURE_OFF)
THEN
o_sdvipc_ADASCarouselModeSelected == 4(CF_ADAPTIVE_CRUISE_CONTROL)

TC_2:
IF
i_can_CarouselModeSwitch == 1(Pressed)
AND
o_sdvipc_ADASCarouselModeSelected == 4(CF_ADAPTIVE_CRUISE_CONTROL)
THEN
o_sdvipc_ADASCarouselModeSelected == 2(CF_SPEED_LIMITER)
______________________________
3
TC_1:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can_i_can_TCS_Deactivated == DEACTIVATED
THEN
o_can_adas_longi_state_comfort == OFF

TC_2:
IF
o_can_adas_longi_state_comfort == ALSC_NOREQUEST_STANDBY(2)
AND
i_can